# Phase III Factorial Experiment Demo

This notebook demonstrates the Phase III factorial experiment using actual Phase I data. The experiment tests 16 combinations of semantic formats (8) and prompting strategies (2) for generating embeddings.


In [44]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

# Import Phase 3 components
from phase3_components import ReferenceRangeManager, FormatGenerator, get_all_format_ids
from phase3_prompts import PromptManager, get_experimental_conditions
from phase3_factorial_experiment import Phase3ExperimentManager


## Load Phase I Data and Preprocessing Objects


In [45]:
# Load preprocessed data and objects
import sys
import warnings
warnings.filterwarnings('ignore')

phase1_dir = Path('../Phase 1/phase_1_outputs')

def load_pickle_safe(filepath):
    """Simple pandas compatibility fix for pickle loading"""
    try:
        with open(filepath, 'rb') as f:
            return pickle.load(f)
    except ModuleNotFoundError:
        # Handle pandas version compatibility
        class CompatibilityModule:
            def __getattr__(self, name):
                return pd.Index if name.endswith('Index') else pd.RangeIndex
        
        sys.modules['pandas.core.indexes.numeric'] = CompatibilityModule()
        try:
            with open(filepath, 'rb') as f:
                return pickle.load(f)
        finally:
            if 'pandas.core.indexes.numeric' in sys.modules:
                del sys.modules['pandas.core.indexes.numeric']

# Load data files
label_encoders = load_pickle_safe(phase1_dir / 'preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42_label_encoders.pkl')
scaler = load_pickle_safe(phase1_dir / 'preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42_scaler.pkl')
X_test = load_pickle_safe(phase1_dir / 'preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42_X_test.pkl')

print(f"Dataset shape: {X_test.shape}")
print(f"Total features: {len(scaler.feature_names_in_)}")

print(f"\nCategorical variables ({len(label_encoders)}):")
for var, encoder_dict in label_encoders.items():
    if isinstance(encoder_dict, dict) and 'classes' in encoder_dict:
        classes = encoder_dict['classes']
        print(f"  {var}: {classes}")
    elif hasattr(encoder_dict, 'classes_'):
        print(f"  {var}: {list(encoder_dict.classes_)}")
    else:
        print(f"  {var}: Could not extract classes from {type(encoder_dict)}")


Dataset shape: (5648, 458)
Total features: 458

Categorical variables (3):
  gender_encoded: ['F', 'M']
  ethnicity_encoded: ['AMERICAN INDIAN/ALASKA NATIVE', 'AMERICAN INDIAN/ALASKA NATIVE FEDERALLY RECOGNIZED TRIBE', 'ASIAN', 'ASIAN - ASIAN INDIAN', 'ASIAN - CAMBODIAN', 'ASIAN - CHINESE', 'ASIAN - FILIPINO', 'ASIAN - JAPANESE', 'ASIAN - KOREAN', 'ASIAN - OTHER', 'ASIAN - THAI', 'ASIAN - VIETNAMESE', 'BLACK/AFRICAN', 'BLACK/AFRICAN AMERICAN', 'BLACK/CAPE VERDEAN', 'BLACK/HAITIAN', 'CARIBBEAN ISLAND', 'HISPANIC OR LATINO', 'HISPANIC/LATINO - CENTRAL AMERICAN (OTHER)', 'HISPANIC/LATINO - COLOMBIAN', 'HISPANIC/LATINO - CUBAN', 'HISPANIC/LATINO - DOMINICAN', 'HISPANIC/LATINO - GUATEMALAN', 'HISPANIC/LATINO - HONDURAN', 'HISPANIC/LATINO - MEXICAN', 'HISPANIC/LATINO - PUERTO RICAN', 'HISPANIC/LATINO - SALVADORAN', 'MIDDLE EASTERN', 'MULTI RACE ETHNICITY', 'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER', 'OTHER', 'PATIENT DECLINED TO ANSWER', 'PORTUGUESE', 'SOUTH AMERICAN', 'UNABLE TO OBTAIN', '

## Prepare Sample Patient Data


In [46]:
# Take first 3 patients for demo
sample_patients = X_test.head(3).copy()

# Unnormalize the scaled features (most features are scaled)
feature_names = scaler.feature_names_in_
scaled_features = [col for col in sample_patients.columns if col in feature_names]

# Create unnormalized data
unnormalized_data = sample_patients.copy()
unnormalized_data[scaled_features] = scaler.inverse_transform(sample_patients[scaled_features])

# For categorical variables, get the original values
for cat_var, encoder in label_encoders.items():
    if cat_var in unnormalized_data.columns:
        # Categorical variables are already encoded as integers, convert back to strings
        try:
            original_values = encoder.inverse_transform(unnormalized_data[cat_var].astype(int))
            unnormalized_data[cat_var] = original_values
        except:
            pass  # Keep as is if inverse transform fails

# Add patient IDs and gender for demo
unnormalized_data['patient_id'] = [f'demo_patient_{i+1:03d}' for i in range(len(unnormalized_data))]
unnormalized_data['gender'] = ['male', 'female', 'male']  # Demo values

print(f"Sample patient features (first 10):")
first_patient = unnormalized_data.iloc[0]
feature_cols = [col for col in unnormalized_data.columns if col not in ['patient_id', 'gender']]
for col in feature_cols[:100]:
    print(f"  {col}: {first_patient[col]:.3f}")
print("  ...")


Sample patient features (first 10):
  age: 48.296
  alanine aminotransferase_mean_count: 3.000
  alanine aminotransferase_mean_last: 48.000
  alanine aminotransferase_mean_mean: 58.333
  alanine aminotransferase_mean_slope_24h: -0.825
  albumin ascites_mean_count: 0.000
  albumin ascites_mean_last: 1.550
  albumin pleural_mean_count: 0.000
  albumin pleural_mean_last: 1.800
  albumin urine_mean_count: 0.000
  albumin urine_mean_last: 6.500
  albumin_mean_count: 2.000
  albumin_mean_last: 2.800
  alkaline phosphate_mean_count: 3.000
  alkaline phosphate_mean_last: 217.000
  alkaline phosphate_mean_mean: 256.333
  alkaline phosphate_mean_slope_24h: -3.792
  anion gap_mean_count: 3.000
  anion gap_mean_last: 15.000
  anion gap_mean_mean: 17.667
  anion gap_mean_slope_24h: -0.163
  anion gap_mean_stddev_24h: 3.055
  asparate aminotransferase_mean_count: 3.000
  asparate aminotransferase_mean_last: 43.000
  asparate aminotransferase_mean_mean: 47.000
  asparate aminotransferase_mean_slope_2

## Initialize Experiment Components


In [47]:
# Initialize the experiment manager
# Note: Reference ranges file may not exist, so we'll handle gracefully
try:
    ref_manager = ReferenceRangeManager("../../data/Lab_reference_ranges.csv")
except FileNotFoundError:
    print("Reference ranges file not found. Creating dummy reference manager.")
    # Create dummy manager for demo
    class DummyReferenceManager:
        def get_interpretation(self, feature_name, value, gender='unknown'):
            return 'Normal'  # Placeholder
    ref_manager = DummyReferenceManager()

format_generator = FormatGenerator(ref_manager)
prompt_manager = PromptManager()

# Get experimental conditions
conditions = get_experimental_conditions()
print(f"Experimental design: {len(conditions)} conditions")
print(f"Format IDs: {get_all_format_ids()}")
print(f"Prompt IDs: {prompt_manager.get_all_prompt_ids()}")


Experimental design: 16 conditions
Format IDs: ['F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8']
Prompt IDs: ['A', 'B']


## Demonstrate Single Patient Processing


In [48]:
# Select first patient
patient_row = unnormalized_data.iloc[0]
patient_id = patient_row['patient_id']
gender = patient_row['gender']
patient_features = patient_row[feature_cols]

print(f"Processing patient: {patient_id} ({gender})")
print(f"Features: {len(patient_features)}")

# Generate all 16 condition outputs
results = {}
for condition in conditions:
    # Generate serialized format
    try:
        serialized_data = format_generator.generate_format(
            patient_data=patient_features,
            format_id=condition['format_id'],
            gender=gender
        )
        
        # Combine with prompt
        full_prompt = prompt_manager.get_full_prompt(
            prompt_id=condition['prompt_id'],
            serialized_data=serialized_data
        )
        
        results[condition['condition_name']] = full_prompt
    except Exception as e:
        results[condition['condition_name']] = f"ERROR: {str(e)}"

print(f"\nGenerated {len(results)} condition outputs")


Processing patient: demo_patient_001 (male)
Features: 458

Generated 16 condition outputs


## Sample Outputs


In [49]:
# Show sample outputs from different conditions
sample_conditions = ['PA_FF1', 'PB_FF1', 'PA_FF3', 'PB_FF8']

for condition_name in sample_conditions:
    if condition_name in results:
        print(f"\n{'='*60}")
        print(f"Condition: {condition_name}")
        print(f"{'='*60}")
        # Show first 300 characters
        preview = results[condition_name][:300] + "..." if len(results[condition_name]) > 300 else results[condition_name]
        print(preview)
        print()
    else:
        print(f"\nCondition {condition_name} not found in results.")
        print(f"Available conditions: {list(results.keys())[:5]}...")



Condition: PA_FF1
Generate an embedding specifically for predicting in-hospital mortality from the following patient data.

age 48.296 alanine aminotransferase_mean_count 3.000 alanine aminotransferase_mean_last 48.000 alanine aminotransferase_mean_mean 58.333 alanine aminotransferase_mean_slope_24h -0.825 alb ascite...


Condition: PB_FF1
You are an experienced ICU physician. Analyze the following data to identify the primary drivers of clinical risk. Generate an embedding that captures the patient's overall severity. First, identify the most abnormal values relative to normal ranges. Second, consider the trends and volatility of the...


Condition: PA_FF3
Generate an embedding specifically for predicting in-hospital mortality from the following patient data.

Abbreviated: age 48.296 alanine aminotransferase_mean_count 3.000 alanine aminotransferase_mean_last 48.000 alanine aminotransferase_mean_mean 58.333 alanine aminotransferase_mean_slope_24h -0.8...


Condition: PB_FF8
You are a

## Process Multiple Patients


In [50]:
# Process all 3 sample patients
all_results = {}

for idx, (_, patient_row) in enumerate(unnormalized_data.iterrows()):
    patient_id = patient_row['patient_id']
    gender = patient_row['gender']
    patient_features = patient_row[feature_cols]
    
    print(f"Processing {patient_id}...")
    
    patient_results = {}
    for condition in conditions[:4]:  # Just first 4 conditions for demo
        try:
            serialized_data = format_generator.generate_format(
                patient_data=patient_features,
                format_id=condition['format_id'],
                gender=gender
            )
            
            full_prompt = prompt_manager.get_full_prompt(
                prompt_id=condition['prompt_id'],
                serialized_data=serialized_data
            )
            
            patient_results[condition['condition_name']] = len(full_prompt)  # Store length for summary
        except Exception as e:
            patient_results[condition['condition_name']] = f"ERROR: {str(e)}"
    
    all_results[patient_id] = patient_results

# Summary of prompt lengths by condition
summary_df = pd.DataFrame(all_results).T
print("\nPrompt lengths by condition:")
print(summary_df)


Processing demo_patient_001...
Processing demo_patient_002...
Processing demo_patient_003...

Prompt lengths by condition:
                  PA_FF1  PA_FF2  PA_FF3  PA_FF4
demo_patient_001   16683   61506   38389   37630
demo_patient_002   16684   61406   38391   37652
demo_patient_003   16685   61397   38393   37630


## Experimental Design Summary


In [51]:
# Print experimental design summary
from phase3_prompts import print_experimental_design

print_experimental_design()


Phase III Factorial Experimental Design
Total Conditions: 16

Prompting Strategies:
--------------------
Prompt A: Task-Specific Predictive
  Description: Direct, concise, and task-aligned prompt establishing clear predictive objective without additional cognitive scaffolding.
  Text: Generate an embedding specifically for predicting in-hospital mortality from the following patient data.

Prompt B: Persona-Driven Diagnostic
  Description: Advanced prompt with expert persona and Chain-of-Thought reasoning structure to enable more nuanced embedding generation.
  Text: You are an experienced ICU physician. Analyze the following data to identify the primary drivers of clinical risk. Generate an embedding that captures the patient's overall severity. First, identify the most abnormal values relative to normal ranges. Second, consider the trends and volatility of these values. Finally, synthesize these factors into the embedding.

Format Combinations:
--------------------
Format F1: Base Dat